# AI-Based Lead Scoring & Analytics System
This notebook trains and evaluates lead conversion models using the UCI Bank Marketing **bank-full.csv** dataset.

**Dataset required:** `../data/bank-full.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, RocCurveDisplay
import joblib
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
DROP_LEAKY_FEATURES = True
LEAKY_COLUMNS = ['duration']  # duration is only known after a call; remove for realistic pre-call scoring


In [ ]:
data_path = Path('../data/bank-full.csv')
if not data_path.exists():
    raise FileNotFoundError('Missing ../data/bank-full.csv. Download bank-full.csv from UCI and place it in lead_scoring_project/data/')
df = pd.read_csv(data_path, sep=';')
print('Loaded shape:', df.shape)
df.head()

In [ ]:
print(df['y'].value_counts(normalize=True))
display(df.describe(include='all').transpose().head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['age'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Age Distribution')
sns.countplot(data=df, x='y', ax=axes[1], palette='viridis')
axes[1].set_title('Target Distribution')
plt.tight_layout()
plt.show()

In [ ]:
df = df.drop_duplicates().copy()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

if DROP_LEAKY_FEATURES:
    df = df.drop(columns=[c for c in LEAKY_COLUMNS if c in df.columns])

print('Columns used:', df.columns.tolist())
df.isna().sum().sum()

In [ ]:
X = df.drop(columns=['y'])
y = (df['y'].str.lower() == 'yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

categorical_cols = X_train.select_dtypes(include='object').columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]
print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, class_weight='balanced')
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'roc_auc']

cv_rows = []
trained_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

    cv_rows.append({
        'model': name,
        'cv_accuracy_mean': scores['test_accuracy'].mean(),
        'cv_precision_mean': scores['test_precision'].mean(),
        'cv_recall_mean': scores['test_recall'].mean(),
        'cv_roc_auc_mean': scores['test_roc_auc'].mean(),
    })

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

cv_results_df = pd.DataFrame(cv_rows).sort_values('cv_roc_auc_mean', ascending=False)
cv_results_df

In [ ]:
best_model_name = cv_results_df.iloc[0]['model']
best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

holdout_metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_prob),
}

print('Best model from CV:', best_model_name)
print('Holdout metrics:', holdout_metrics)
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title(f'ROC Curve - {best_model_name} (Holdout)')
plt.show()

In [ ]:
model_bundle = {
    'model': best_model,
    'feature_columns': X.columns.tolist(),
    'selected_model': best_model_name,
    'cv_metrics': cv_results_df.to_dict(orient='records'),
    'holdout_metrics': holdout_metrics,
    'dataset': 'UCI Bank Marketing - bank-full.csv',
}
joblib.dump(model_bundle, '../backend/model.pkl')
print('Saved model to ../backend/model.pkl')